Project Koban: env used will be ml_env_tf_2.15
Goals of this project: 
    Develop a pipeline for image segmentation with minimal assistance from AI agents. 
Structure:
    Object Loading: Object classes (Images, u-net models)
        Considerations: hyperstacks can be very large and memory heavy. I need to use efficent loading methods such as memmap loading to ensure I don't run out of memory. 
        Pathing: flexible pathing that can acomidate local windows or linux system. 
                Points to where the object being loaded lives on the current system. Paths will be defined as variables which will then be called inside of functions. 
                Returns: windows vs linux, file exists. 
                Packages required: Pathlib, os 

        Images: 5D Hyperstacks (dimentions may vary so I should build such that meta data defines the image and the pipeline is flexible to acommodate different dimentions)
            Returns: Image Shape, Dimentions, axes,
                        Expected image shape T,C,Z,X,Y  
                Packages: Tifffile, numpy
        
        U-net Models: 3 class models containing 0 = background, 1 = nucleus, 2 = droplets. .karas format
               Returns: A 512 x 512 probility array 
                Set model path to load. 
            

    Image Segmentation: 
        U-net Classification: 
            Segmentation: u-net model applies to the image.
            Constraints: 

In [ ]:
print("Kernal Alive")

In [ ]:
#### Pakages###
import tifffile as tiff
import numpy as np
import pathlib
import os
from dataclasses import dataclass, field
import platform
import datetime
from pathlib import Path

try:
    tensorflow as tf
except Exception:
    tf = None 

In [ ]:
#### Configuration Section#### 
#------------------------------------------------
# System Configurations Set Mainly for HPC Use
#------------------------------------------------
def get_n_workers() -> int:
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))
def config_tensorflow_for_GPU(config:PipelineConfig) -> None:
    if tf is None:
        print("tensorflow not avalible - skipping gpu setup")
        return 

#------------------------------------------------
# Smart Root Pathing for HPC and Local Use
#------------------------------------------------
def get_system() -> str: 
    system = platform.system()
    if system == 'Windows':
        return 'windows'
    elif system == 'Linux':
        return 'linux'
    else:
        raise RuntimeError(f"Unsupported system: {system}")

system = get_system()
print(system)

# System dependant DATA_ROOT
if system == 'windows':
    DATA_ROOT = pathlib.Path("c:/users/cowboy/OneDrive - UAB - The University of Alabama at Birmingham")
elif system == 'linux':
    DATA_ROOT = pathlib.Path("/home/tdeibert/")
else: 
    raise RuntimeError(f"Unsupported system: {system}")

# Sanity Check to Ensure ROOT exists
if DATA_ROOT.exists():
    print("This Is The Way")
else: 
    raise RuntimeError(f"Path not Found: {DATA_ROOT}")

#------------------------------------------------
# Experimental Configurations Set per Experiment
#------------------------------------------------
@dataclass 
class ExperimentConfig:
    #### Experiment Specific Items #### 
    experiment_name: str = "Control" 
    treatment: str = "none"
    replicate: str = "One"
    channel_map: dict[str,int] = field(default_factory =lambda: {"DIO_Membranes": 0, "mCherry_NLS": 1, "Alexaflour647_NPC": 2})
    pixel_size: float = 0.108  #objective dependant
    z_step_size: float = 1.0 
    folder_name: Path = pathlib.Path("/Nuclear_Scaling/Data/Control")
    subfolder_name: Path = pathlib.Path("Extract_One")
    image_name: Path = pathlib.Path("control_extract_1.tif")
    experiment_date: datetime.date = datetime.date(2026, 5, 6) #optional for record keeping
    @property
    def image_path(self) -> Path:
        return Path(DATA_ROOT/self.folder_name/self.subfolder_name/self.image_name)

#------------------------------------------------
# Pipeline Configurations Set per Pipeline
#------------------------------------------------
@dataclass
class PipelineConfig:
    #### Pathing ####
    model_root: Path = pathlib.Path("Code_Repo/Nuclear_Scaling_ML/Python_Projects/Trained_Models/")
    model: Path = pathlib.Path("unet_droplet_nucleus_v6_2_best.keras")
    @property
    def model_path(self) -> Path:
        return Path(DATA_ROOT/self.model_root/self.model)

windows
This Is The Way


Placeholder markdown